In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#1. Load all the .txt files

files = [
    "hospital_policy.txt",
    "paracetamol_info.txt",
    "diabetes_factsheet.txt",
    "insulin_guide.txt",
    "sample_emr.txt"
]

docs=[]

for filename in files:
    docs.extend(TextLoader(filename).load())

print(f"Loaded {len(docs)} documents. Example excerpt :\n", docs[0].page_content[:200])

Loaded 5 documents. Example excerpt :
 Hospital Discharge Policy

All inpatients must be reviewed daily by their primary physician. Patients are considered for discharge when their primary medical condition is stable, they no longer requir


In [4]:
#2. Chunk with metadata
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

chunks= splitter.split_documents(docs)
for i,chunk in enumerate(chunks):
    chunk.metadata["source"]= chunk.metadata.get("source", files[0])
    chunk.metadata["chunk_id"]= i+1

print(f"Total chunks: {len(chunks)}")

Total chunks: 12


In [5]:
#3. Embedding
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [6]:
#4. Vector Store( FAISS, Local and Free)

from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embedding = embeddings)

In [7]:
#1. Save FAISS vector store (after building)

vectorstore.save_local("faiss_index_healthcare")

In [11]:
#5. Retrieval Chain(LCEL Style)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI

system_message = SystemMessagePromptTemplate.from_template("""
You are a helpful healthcare assistant. Answer clearly and patiently.
If the answer is not present in the provided context, reply: 'Sorry, I do not have information'
""")

human_message = HumanMessagePromptTemplate.from_template("""
Use the following context to answer the user's question:
{context}

Question: {input}
Answer:
""")

prompt = ChatPromptTemplate.from_messages([system_message,human_message])
llm = ChatOpenAI(model="gpt-4o-mini", temperature =0)

combine_docs_chain = create_stuff_documents_chain(llm, prompt)

In [12]:
from langchain_core.runnables import RunnableMap, RunnableLambda

qa_chain =(
    RunnableMap({
        "input" : lambda x: x['input'],
        "context": lambda x: retriever.invoke(x['input'])
    })
    |
     RunnableLambda(lambda inputs: {
        "answer": combine_docs_chain.invoke(inputs),
        "source_documents": inputs["context"]
     })
)

In [13]:
questions = [
    "What are the side effects of paracetamol?",
    "What is the discharge policy for ICU patients?",
    "How is diabetes managed as per our hospital’s guidelines?",
    "How do I take insulin?"
]

for q in questions:
    result = qa_chain.invoke({"input": q})
    print(f"\nQ: {q}\nA: {result['answer']}\n")
    print("--- Source chunk(s) ---")
    for doc in result["source_documents"]:
        print(f"[source: {doc.metadata.get('source')}, chunk {doc.metadata.get('chunk_id')}]")
        print(doc.page_content, "\n")


Q: What are the side effects of paracetamol?
A: The side effects of paracetamol include rare occurrences of rash, low blood pressure, or liver injury, which usually happens with overdose or prolonged use.

--- Source chunk(s) ---
[source: paracetamol_info.txt, chunk 5]
**Side Effects**: Paracetamol is well tolerated. Rare side effects include rash, low blood pressure, or liver injury (usually with overdose or prolonged use). In case of overdose, seek medical help immediately. 

[source: paracetamol_info.txt, chunk 4]
Paracetamol (Acetaminophen) Information

Paracetamol is a commonly used pain reliever and fever reducer. It is suitable for adults and children. Usual adult dose is 500mg to 1000mg every 4-6 hours, not exceeding 4g per day. 

[source: paracetamol_info.txt, chunk 6]
**Precautions**: Avoid taking more than the recommended dose. People with liver disease should consult their doctor before use. Do not combine with other products containing acetaminophen.

For more information